In [500]:
import pandas as pd
import pycountry
import ast
import random

In [501]:
df = pd.read_csv("../QA/QA_final.csv")

In [502]:
to_maintain = ["ciudadania", "fecha de nacimiento", "ocupacion", "fecha de defuncion", "lugar de nacimiento"]

In [503]:
df_to_maintain = df[df["relacion"].isin(to_maintain)]

In [504]:
df_to_maintain = df_to_maintain[["entidad", "relacion", "objetos"]].reset_index(drop=True)

In [505]:
def tomar_primer_objeto(x):
    lista = ast.literal_eval(x)
    return lista[0]

def formatear(p):
    return p.replace(".", "").replace(",", "").replace(" ", "_").replace("-", "_").title()

def only_year(date):
    return str(date).split("-")[0]

In [506]:
df_to_maintain["entidad"] = df_to_maintain["entidad"].apply(formatear)

In [507]:
df_to_maintain["objetos"] = df_to_maintain["objetos"].apply(tomar_primer_objeto)

In [508]:
paises = {c.name.lower() for c in pycountry.countries}

def es_pais(x):
    country = x.lower()
    return country if country in paises else None

In [509]:
mask = df_to_maintain["relacion"].isin(["lugar de nacimiento", "ciudadania"])
df_to_maintain.loc[mask, "objetos"] = df_to_maintain.loc[mask, "objetos"].apply(es_pais)
df_to_maintain = df_to_maintain[~(mask & df_to_maintain["objetos"].isna())].copy()
df_to_maintain.loc[mask, "objetos"] = df_to_maintain.loc[mask, "objetos"].apply(formatear)

In [510]:
mask = df_to_maintain["relacion"].isin(["fecha de nacimiento", "fecha de defuncion"])
df_to_maintain.loc[mask, "objetos"] = df_to_maintain.loc[mask, "objetos"].apply(only_year)

In [511]:
mask = df_to_maintain["relacion"].isin(["ocupacion"])
df_to_maintain.loc[mask, "objetos"] = df_to_maintain.loc[mask, "objetos"].apply(formatear)

In [512]:
df_to_maintain = df_to_maintain[df_to_maintain["entidad"].str.count("_") <= 1]
df_to_maintain = df_to_maintain[df_to_maintain["objetos"].str.count("_") <= 1]

In [513]:
mask_no_links = ~df_to_maintain["entidad"].str.contains("http://", na=False)
df_to_maintain = df_to_maintain[mask_no_links].copy()

mask_no_links = ~df_to_maintain["objetos"].str.contains("http://", na=False)
df_to_maintain = df_to_maintain[mask_no_links].copy()

In [514]:
df_to_maintain.shape

(17041, 3)

In [515]:
rel_to_values = (
    df_to_maintain
    .groupby("relacion")["objetos"]
    .unique()
    .to_dict()
)

In [516]:
def sample_different_object(row):
    rel = row["relacion"]
    obj = row["objetos"]
    values = rel_to_values[rel]

    candidates = [v for v in values if v != obj]

    return random.choice(candidates)

In [517]:
df_to_maintain["objeto_falso"] = df_to_maintain.apply(sample_different_object, axis=1)

In [518]:
relation_map = {
    "ciudadania": " posee la ciudadanía del país ",
    "fecha de nacimiento": " nació en el año ",
    "ocupacion": " se desempeña como ",
    "fecha de defuncion": " falleció en el año ",
    "lugar de nacimiento": " nació en ",
}

In [519]:
def formulate_context(row):
    relation = relation_map[row["relacion"]]
    return row["entidad"] + relation + row["objeto_falso"]

In [520]:
df_to_maintain["context_line"] = df_to_maintain.apply(formulate_context, axis=1)

In [521]:
question_templates = {
    "ciudadania": "¿De qué país posee la ciudadanía {}?",
    "fecha de nacimiento": "¿En qué año nació {}?",
    "ocupacion": "¿En qué se desempeña {}?",
    "fecha de defuncion": "¿En qué año falleció {}?",
    "lugar de nacimiento": "¿En qué país nació {}?",
}

In [522]:
def build_question(relacion, entidad):
    template = question_templates[relacion]
    return template.format(entidad)

In [523]:
def create_binding_example(df, relacion, n=3):
    subset = df[df["relacion"] == relacion]
    samples = subset.sample(n)
    target_row = samples.sample(1).iloc[0]
    
    context = "\n".join(samples["context_line"].tolist())
    
    pregunta = f"Pregunta: {build_question(relacion, target_row['entidad'])}"
    
    respuesta = target_row["objeto_falso"]
    
    prompt = (
        "Responde la pregunta basándote en el siguiente contexto. Mantén la respuesta breve.\n\n"
        "--- Contexto ---\n"
        f"{context}\n\n"
        "--- Pregunta ---\n"
        f"{pregunta}\n\n"
    )

    return prompt, respuesta


In [525]:
context_length = 2
n_examples = 100

for context_length in range(2, 4):
    for relacion_a_generar in to_maintain:
        rows = []
        
        for i in range(n_examples):
            example_text, respuesta = create_binding_example(df_to_maintain, relacion=relacion_a_generar, n=context_length)

            rows.append({
                "prompt": example_text,
                "respuesta": respuesta
            })

        df_out = pd.DataFrame(rows)
        df_out.to_csv(f"context_length_{context_length}/{relacion_a_generar}.csv", index=False, encoding="utf-8")
